## original fertility test

In [ ]:
import random
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import WhitespaceSplit

# Load tokenizer
TOKENIZER_PATH = Path("../shared_tokenizer2/tokenizer.json").resolve()
assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
word_splitter = WhitespaceSplit()

DATASETS = {
    "English": "BabyLM-community/babylm-eng",
    "Dutch": "BabyLM-community/babylm-nld",
    "Indonesian": "BabyLM-community/babylm-ind",
    "Javanese": "BabyLM-community/babylm-jav",
}

MAX_WORDS_PER_LANG = 300_000

# 每个 chunk 的大小
CHUNK_WORDS = 5_000
SEED = 42


def count_tokens(text):
    # 重要：分 chunk 算的时候，不要给每个 chunk 加 special tokens
    return len(tokenizer.encode(text, add_special_tokens=False).ids)


def get_text_column(dataset):
    candidates = ["text", "sentence", "content", "raw_text"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Cannot find text column. Available columns: {dataset.column_names}")


def load_best_split(dataset_name):
    ds_dict = load_dataset(dataset_name)
    for split in ["validation", "valid", "dev", "test", "train"]:
        if split in ds_dict:
            return ds_dict[split], split
    first_split = list(ds_dict.keys())[0]
    return ds_dict[first_split], first_split


def extract_chunk_by_word_offsets(text, word_offsets, start_word_idx, num_words):
    """
    从 text 里按照 WhitespaceSplit word offset 抽取一个 chunk。
    这样比 ' '.join(words) 更接近原始文本。
    """
    end_word_idx = min(start_word_idx + num_words, len(word_offsets))

    start_char = word_offsets[start_word_idx][1][0]
    end_char = word_offsets[end_word_idx - 1][1][1]

    chunk_text = text[start_char:end_char]
    actual_words = end_word_idx - start_word_idx

    return chunk_text, actual_words


results = []

for lang, dataset_name in DATASETS.items():
    print(f"\nLoading {lang}: {dataset_name}")

    ds, split_name = load_best_split(dataset_name)
    text_col = get_text_column(ds)

    # 先 shuffle example 顺序；对于 English 这种只有一个超长 example 的情况，后面还会在内部抽 chunks
    ds = ds.shuffle(seed=SEED)

    rng = random.Random(SEED)

    total_words = 0
    total_tokens = 0
    sampled_segments = 0
    source_examples = 0

    for example in ds:
        if total_words >= MAX_WORDS_PER_LANG:
            break

        text = example[text_col]

        if not isinstance(text, str):
            continue

        text = text.strip()
        if not text:
            continue

        # 得到每个 WhitespaceSplit word 在原始文本中的位置
        word_offsets = word_splitter.pre_tokenize_str(text)
        n_words = len(word_offsets)

        if n_words == 0:
            continue

        source_examples += 1

        # 把这个 example 切成很多 chunk 的起点
        chunk_starts = list(range(0, n_words, CHUNK_WORDS))

        # 关键：随机打乱 chunk 起点，不要只取文本开头
        rng.shuffle(chunk_starts)

        for start_idx in chunk_starts:
            if total_words >= MAX_WORDS_PER_LANG:
                break

            remaining_words = MAX_WORDS_PER_LANG - total_words
            words_to_take = min(CHUNK_WORDS, remaining_words, n_words - start_idx)

            if words_to_take <= 0:
                continue

            chunk_text, words_used = extract_chunk_by_word_offsets(
                text=text,
                word_offsets=word_offsets,
                start_word_idx=start_idx,
                num_words=words_to_take,
            )

            tokens_used = count_tokens(chunk_text)

            total_words += words_used
            total_tokens += tokens_used
            sampled_segments += 1

    if total_words < MAX_WORDS_PER_LANG:
        print(
            f"Warning: {lang} only has {total_words} words, "
            f"less than target {MAX_WORDS_PER_LANG}."
        )

    fertility = total_tokens / total_words

    results.append({
        "language": lang,
        "dataset": dataset_name,
        "split": split_name,
        "source_examples": source_examples,
        "sampled_segments": sampled_segments,
        "words": total_words,
        "tokens": total_tokens,
        "fertility": fertility,
        "words_per_1M_tokens": 1_000_000 / fertility,
    })


df = pd.DataFrame(results)

eng_fertility = df.loc[df["language"] == "English", "fertility"].iloc[0]
df["tokenization_tax_vs_English"] = df["fertility"] / eng_fertility - 1

print("\n=== Fertility Results ===")
print(df.to_string(index=False))

### check the data quality of JAV: padding & child books

In [ ]:
from datasets import load_dataset
from tokenizers.pre_tokenizers import WhitespaceSplit
import pandas as pd

word_splitter = WhitespaceSplit()

def get_text_column(dataset):
    candidates = ["text", "sentence", "content", "raw_text"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Cannot find text column. Available columns: {dataset.column_names}")

ds = load_dataset("BabyLM-community/babylm-jav", split="train")
text_col = get_text_column(ds)

rows = []

for ex in ds:
    text = ex[text_col]
    if not isinstance(text, str):
        continue

    text = text.strip()
    if not text:
        continue

    n_words = len(word_splitter.pre_tokenize_str(text))

    row = {"n_words": n_words}

    # 如果 dataset 有 category，比如 child-books / padding，也一起记录
    if "category" in ds.column_names:
        row["category"] = ex["category"]

    rows.append(row)

df_len = pd.DataFrame(rows)

print("Total examples:", len(df_len))
print("Total words:", df_len["n_words"].sum())
print("\nWord length statistics:")
print(df_len["n_words"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

if "category" in df_len.columns:
    print("\nWords by category:")
    print(df_len.groupby("category")["n_words"].agg(["count", "sum", "mean", "median"]))

In [ ]:
from collections import defaultdict
from datasets import load_dataset
from tokenizers.pre_tokenizers import WhitespaceSplit
from tokenizers import Tokenizer
from pathlib import Path
import random

TOKENIZER_PATH = Path("../shared_tokenizer2/tokenizer.json").resolve()
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
word_splitter = WhitespaceSplit()

MAX_WORDS_PER_LANG = 300_000
CHUNK_WORDS = 5_000
SEED = 42

ds = load_dataset("BabyLM-community/babylm-jav", split="train")
ds = ds.shuffle(seed=SEED)

text_col = "text"
rng = random.Random(SEED)

category_stats = defaultdict(lambda: {
    "source_examples": 0,
    "sampled_segments": 0,
    "words": 0,
    "tokens": 0,
})

total_words = 0
total_tokens = 0

for example in ds:
    if total_words >= MAX_WORDS_PER_LANG:
        break

    text = example[text_col]
    if not isinstance(text, str) or not text.strip():
        continue

    text = text.strip()
    word_offsets = word_splitter.pre_tokenize_str(text)
    n_words = len(word_offsets)

    if n_words == 0:
        continue

    category = example["category"] if "category" in ds.column_names else "unknown"
    example_used = False

    chunk_starts = list(range(0, n_words, CHUNK_WORDS))
    rng.shuffle(chunk_starts)

    for start_idx in chunk_starts:
        if total_words >= MAX_WORDS_PER_LANG:
            break

        remaining_words = MAX_WORDS_PER_LANG - total_words
        words_to_take = min(CHUNK_WORDS, remaining_words, n_words - start_idx)

        if words_to_take <= 0:
            continue

        end_word_idx = start_idx + words_to_take
        start_char = word_offsets[start_idx][1][0]
        end_char = word_offsets[end_word_idx - 1][1][1]

        chunk_text = text[start_char:end_char]
        tokens_used = len(tokenizer.encode(chunk_text).ids)

        total_words += words_to_take
        total_tokens += tokens_used

        category_stats[category]["sampled_segments"] += 1
        category_stats[category]["words"] += words_to_take
        category_stats[category]["tokens"] += tokens_used

        if not example_used:
            category_stats[category]["source_examples"] += 1
            example_used = True

print("Total sampled words:", total_words)
print("Total sampled tokens:", total_tokens)
print("Fertility:", total_tokens / total_words)

print("\nCategory breakdown:")
for cat, stats in category_stats.items():
    stats["fertility"] = stats["tokens"] / stats["words"]
    stats["word_share"] = stats["words"] / total_words
    print(cat, stats)

## fertility - only child books(JAV)

In [ ]:
import random
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import WhitespaceSplit

# Load tokenizer
TOKENIZER_PATH = Path("../shared_tokenizer2/tokenizer.json").resolve()
assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
word_splitter = WhitespaceSplit()

DATASETS = {
    "English": "BabyLM-community/babylm-eng",
    "Dutch": "BabyLM-community/babylm-nld",
    "Indonesian": "BabyLM-community/babylm-ind",
    "Javanese": "BabyLM-community/babylm-jav",
}

MAX_WORDS_PER_LANG = 300_000

# 每个 chunk 的大小
CHUNK_WORDS = 5_000
SEED = 42


def count_tokens(text):
    # 重要：分 chunk 算的时候，不要给每个 chunk 加 special tokens
    return len(tokenizer.encode(text, add_special_tokens=False).ids)


def get_text_column(dataset):
    candidates = ["text", "sentence", "content", "raw_text"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Cannot find text column. Available columns: {dataset.column_names}")


def load_best_split(dataset_name):
    ds_dict = load_dataset(dataset_name)
    for split in ["validation", "valid", "dev", "test", "train"]:
        if split in ds_dict:
            return ds_dict[split], split
    first_split = list(ds_dict.keys())[0]
    return ds_dict[first_split], first_split


def extract_chunk_by_word_offsets(text, word_offsets, start_word_idx, num_words):
    """
    从 text 里按照 WhitespaceSplit word offset 抽取一个 chunk。
    这样比 ' '.join(words) 更接近原始文本。
    """
    end_word_idx = min(start_word_idx + num_words, len(word_offsets))

    start_char = word_offsets[start_word_idx][1][0]
    end_char = word_offsets[end_word_idx - 1][1][1]

    chunk_text = text[start_char:end_char]
    actual_words = end_word_idx - start_word_idx

    return chunk_text, actual_words


results = []

for lang, dataset_name in DATASETS.items():
    print(f"\nLoading {lang}: {dataset_name}")

    ds, split_name = load_best_split(dataset_name)
    text_col = get_text_column(ds)

    # Only for Javanese: use child-books subset, excluding padding
    if lang == "Javanese":
        assert "category" in ds.column_names, "Javanese dataset has no category column."
        ds = ds.filter(lambda x: x["category"] == "child-books")
        print("Filtered Javanese to child-books only.")
        print("Remaining examples:", len(ds))

    ds = ds.shuffle(seed=SEED)

    rng = random.Random(SEED)

    total_words = 0
    total_tokens = 0
    sampled_segments = 0
    source_examples = 0

    for example in ds:
        if total_words >= MAX_WORDS_PER_LANG:
            break

        text = example[text_col]

        if not isinstance(text, str):
            continue

        text = text.strip()
        if not text:
            continue

        # 得到每个 WhitespaceSplit word 在原始文本中的位置
        word_offsets = word_splitter.pre_tokenize_str(text)
        n_words = len(word_offsets)

        if n_words == 0:
            continue

        source_examples += 1

        # 把这个 example 切成很多 chunk 的起点
        chunk_starts = list(range(0, n_words, CHUNK_WORDS))

        # 关键：随机打乱 chunk 起点，不要只取文本开头
        rng.shuffle(chunk_starts)

        for start_idx in chunk_starts:
            if total_words >= MAX_WORDS_PER_LANG:
                break

            remaining_words = MAX_WORDS_PER_LANG - total_words
            words_to_take = min(CHUNK_WORDS, remaining_words, n_words - start_idx)

            if words_to_take <= 0:
                continue

            chunk_text, words_used = extract_chunk_by_word_offsets(
                text=text,
                word_offsets=word_offsets,
                start_word_idx=start_idx,
                num_words=words_to_take,
            )

            tokens_used = count_tokens(chunk_text)

            total_words += words_used
            total_tokens += tokens_used
            sampled_segments += 1

    if total_words < MAX_WORDS_PER_LANG:
        print(
            f"Warning: {lang} only has {total_words} words, "
            f"less than target {MAX_WORDS_PER_LANG}."
        )

    fertility = total_tokens / total_words

    results.append({
        "language": lang,
        "dataset": dataset_name,
        "split": split_name,
        "source_examples": source_examples,
        "sampled_segments": sampled_segments,
        "words": total_words,
        "tokens": total_tokens,
        "fertility": fertility,
        "words_per_1M_tokens": 1_000_000 / fertility,
    })


df = pd.DataFrame(results)

eng_fertility = df.loc[df["language"] == "English", "fertility"].iloc[0]
df["tokenization_tax_vs_English"] = df["fertility"] / eng_fertility - 1

print("\n=== Fertility Results ===")
print(df.to_string(index=False))

In [ ]:
# Corrected fertility without padding: Javanese child-books only
# Word count is aligned with train.py's tokenizer pre-tokenizer: WhitespaceSplit().

import random
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.pre_tokenizers import WhitespaceSplit

TOKENIZER_PATH = Path("../shared_tokenizer2/tokenizer.json").resolve()
assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
word_splitter = WhitespaceSplit()

DATASETS = {
    "English": "BabyLM-community/babylm-eng",
    "Dutch": "BabyLM-community/babylm-nld",
    "Indonesian": "BabyLM-community/babylm-ind",
    "Javanese": "BabyLM-community/babylm-jav",
}

MAX_WORDS_PER_LANG = 300_000
CHUNK_WORDS = 5_000
SEED = 42


def count_tokens(text):
    # Do not add special tokens when counting tokenization fertility.
    return len(tokenizer.encode(text, add_special_tokens=False).ids)


def get_text_column(dataset):
    candidates = ["text", "sentence", "content", "raw_text"]
    for col in candidates:
        if col in dataset.column_names:
            return col
    raise ValueError(f"Cannot find text column. Available columns: {dataset.column_names}")


def load_best_split(dataset_name):
    ds_dict = load_dataset(dataset_name)
    for split in ["validation", "valid", "dev", "test", "train"]:
        if split in ds_dict:
            return ds_dict[split], split
    first_split = list(ds_dict.keys())[0]
    return ds_dict[first_split], first_split


def extract_chunk_by_word_offsets(text, word_offsets, start_word_idx, num_words):
    end_word_idx = min(start_word_idx + num_words, len(word_offsets))
    start_char = word_offsets[start_word_idx][1][0]
    end_char = word_offsets[end_word_idx - 1][1][1]
    chunk_text = text[start_char:end_char]
    actual_words = end_word_idx - start_word_idx
    return chunk_text, actual_words


results = []

for lang, dataset_name in DATASETS.items():
    print(f"\nLoading {lang}: {dataset_name}")

    ds, split_name = load_best_split(dataset_name)
    text_col = get_text_column(ds)

    # Without padding: only remove the Javanese padding rows.
    if lang == "Javanese":
        assert "category" in ds.column_names, "Javanese dataset has no category column."
        ds = ds.filter(lambda x: x["category"] == "child-books")
        print("Filtered Javanese to child-books only.")
        print("Remaining examples:", len(ds))

    ds = ds.shuffle(seed=SEED)
    rng = random.Random(SEED)

    total_words = 0
    total_tokens = 0
    sampled_segments = 0
    source_examples = 0

    for example in ds:
        if total_words >= MAX_WORDS_PER_LANG:
            break

        text = example[text_col]
        if not isinstance(text, str):
            continue

        text = text.strip()
        if not text:
            continue

        word_offsets = word_splitter.pre_tokenize_str(text)
        n_words = len(word_offsets)
        if n_words == 0:
            continue

        source_examples += 1
        chunk_starts = list(range(0, n_words, CHUNK_WORDS))
        rng.shuffle(chunk_starts)

        for start_idx in chunk_starts:
            if total_words >= MAX_WORDS_PER_LANG:
                break

            remaining_words = MAX_WORDS_PER_LANG - total_words
            words_to_take = min(CHUNK_WORDS, remaining_words, n_words - start_idx)
            if words_to_take <= 0:
                continue

            chunk_text, words_used = extract_chunk_by_word_offsets(
                text=text,
                word_offsets=word_offsets,
                start_word_idx=start_idx,
                num_words=words_to_take,
            )
            tokens_used = count_tokens(chunk_text)

            total_words += words_used
            total_tokens += tokens_used
            sampled_segments += 1

    if total_words < MAX_WORDS_PER_LANG:
        print(
            f"Warning: {lang} only has {total_words} words, "
            f"less than target {MAX_WORDS_PER_LANG}."
        )

    fertility = total_tokens / total_words
    results.append({
        "language": lang,
        "dataset": dataset_name,
        "split": split_name,
        "source_examples": source_examples,
        "sampled_segments": sampled_segments,
        "words": total_words,
        "tokens": total_tokens,
        "fertility": fertility,
        "words_per_1M_tokens": 1_000_000 / fertility,
    })


df_without_padding = pd.DataFrame(results)
eng_fertility = df_without_padding.loc[
    df_without_padding["language"] == "English", "fertility"
].iloc[0]
df_without_padding["tokenization_tax_vs_English"] = (
    df_without_padding["fertility"] / eng_fertility - 1
)

print("\n=== Corrected Fertility Results: without padding ===")
print(df_without_padding.to_string(index=False))
df_without_padding
